## Load Data and Split

In [1]:
import os, sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Add project root to path
project_root = "/Users/cindychen/Desktop/EE562/EE562 Assignments/EE562-Classifiers"
if project_root not in sys.path:
    sys.path.append(project_root)

In [8]:
# split the dataset and save it to the data folder
from src.datasets.doggie_loader import split_stanford_dogs
if __name__ == "__main__":
    split_stanford_dogs()

n02097658-silky_terrier: 170 train, 3 val, 10 test
n02092002-Scottish_deerhound: 216 train, 4 val, 12 test
n02099849-Chesapeake_Bay_retriever: 155 train, 3 val, 9 test
n02091244-Ibizan_hound: 175 train, 3 val, 10 test
n02095314-wire-haired_fox_terrier: 146 train, 2 val, 9 test
n02091831-Saluki: 186 train, 3 val, 11 test
n02102318-cocker_spaniel: 148 train, 3 val, 8 test
n02104365-schipperke: 143 train, 2 val, 9 test
n02090622-borzoi: 140 train, 2 val, 9 test
n02113023-Pembroke: 168 train, 3 val, 10 test
n02105505-komondor: 143 train, 2 val, 9 test
n02093256-Staffordshire_bullterrier: 144 train, 2 val, 9 test
n02113799-standard_poodle: 148 train, 3 val, 8 test
n02109961-Eskimo_dog: 139 train, 2 val, 9 test
n02089973-English_foxhound: 146 train, 2 val, 9 test
n02099601-golden_retriever: 139 train, 2 val, 9 test
n02095889-Sealyham_terrier: 188 train, 3 val, 11 test
n02085782-Japanese_spaniel: 172 train, 3 val, 10 test
n02097047-miniature_schnauzer: 143 train, 2 val, 9 test
n02110063-malam

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import random
from pathlib import Path
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from torchvision import datasets
from torch.utils.data import DataLoader

## Run CNN

Importing all the functions from the resnet file

In [3]:
# importing the functions from the resnet module
from src.models.resnet_cnn import (
    create_resnet18,
    get_default_transforms,
    train_epoch,
    validate_epoch,
    predict_image,
    get_all_predictions
)

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


In [4]:
# setting path and loading the dataset

data_path = "/Users/cindychen/Desktop/EE562/EE562 Assignments/EE562-Classifiers/data/dataset_split"
train_transform, val_transform = get_default_transforms()

# Loading the dataset:
train_dataset = datasets.ImageFolder(root=Path(data_path)/'train', transform=train_transform)
val_dataset = datasets.ImageFolder(root=Path(data_path)/'val', transform=val_transform)
test_dataset = datasets.ImageFolder(root=Path(data_path)/'test', transform=val_transform)

class_names = train_dataset.classes
num_classes = len(class_names)

print(f"Number of classes: {num_classes}")
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}") 
print(f"Test samples: {len(test_dataset)}")

# the ratio's used here for train, val and testing are made to be the same as the ratio of the other dataset, such that training is equal.

Number of classes: 120
Train samples: 19116
Val samples: 316
Test samples: 1148


In [5]:
# creating data loaders for training, validation and testing
batch_size = 16 # changing depending on GPU
print(f"\nUsing batch size: {batch_size}")

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=batch_size, 
    shuffle=False,
    num_workers=0
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")


Using batch size: 16
Train batches: 1195
Validation batches: 20
Test batches: 72


In [6]:
# start the model
model = create_resnet18(num_classes=num_classes)

# model parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"trainable parameters: {trainable_params:,}")


Total parameters: 11,500,728
trainable parameters: 11,500,728


In [ ]:
#  Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.1, patience=3
)

print(f"Loss function: CrossEntropyLoss")
print(f"Optimizer: Adam (lr=0.001, weight_decay=1e-4)")
print(f"Scheduler: ReduceLROnPlateau")

Loss function: CrossEntropyLoss
Optimizer: Adam (lr=0.001, weight_decay=1e-4)
Scheduler: ReduceLROnPlateau


In [9]:
# Training loop
num_epochs = 10
best_val_acc = 0.0

# Lists to store metrics
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 30)
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    val_loss, val_acc = validate_epoch(model, val_loader, criterion)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    scheduler.step(val_loss)
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_dog_model.pth')
        print(f"✓ Saved best model (val_acc: {val_acc:.2f}%)")


Epoch 1/10
------------------------------


Training:   0%|          | 0/1195 [00:00<?, ?it/s]

Validation: 100%|██████████| 20/20 [00:08<00:00,  2.44it/s, Loss=4.8134, Acc=1.27%]


Train Loss: 4.7939, Train Acc: 1.02%
Val Loss: 4.7797, Val Acc: 1.27%
✓ Saved best model (val_acc: 1.27%)

Epoch 2/10
------------------------------


Validation: 100%|██████████| 20/20 [00:08<00:00,  2.43it/s, Loss=4.8316, Acc=1.27%]


...

Train Loss: 4.6150, Train Acc: 1.83%
Val Loss: 4.5513, Val Acc: 1.58%

Epoch 7/10
------------------------------


Validation: 100%|██████████| 20/20 [00:16<00:00,  1.21it/s, Loss=4.6626, Acc=2.85%]


Train Loss: 4.5518, Train Acc: 2.42%
Val Loss: 4.5006, Val Acc: 2.85%
✓ Saved best model (val_acc: 2.85%)

Epoch 8/10
------------------------------


Training:  82%|████████▏ | 976/1195 [27:44<06:13,  1.71s/it, Loss=4.3226, Acc=2.98%]  


KeyboardInterrupt: 

In [ ]:
# Cell 7: Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, 'b-o', label='Train Loss', markersize=4)
ax1.plot(val_losses, 'r-s', label='Val Loss', markersize=4)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(train_accs, 'b-o', label='Train Accuracy', markersize=4)
ax2.plot(val_accs, 'r-s', label='Val Accuracy', markersize=4)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

In [ ]:
# Cell 8: Test evaluation
print("="*50)
print("TEST SET EVALUATION")
print("="*50)

model.load_state_dict(torch.load('best_dog_model.pth'))
test_loss, test_acc = validate_epoch(model, test_loader, criterion)
print(f"\nFinal Test Results:")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test Accuracy: {test_acc:.2f}%")

In [ ]:
# Cell 9: Confusion Matrix
test_preds, test_labels = get_all_predictions(model, test_loader)
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=False, cmap='Blues', cbar=True)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

class_acc = cm.diagonal() / cm.sum(axis=1)
print(f"Mean class accuracy: {np.nanmean(class_acc):.4f}")

In [ ]:
# Cell 10: Single prediction example
test_idx = random.randint(0, len(test_dataset) - 1)
img_path, true_label = test_dataset.samples[test_idx]
true_breed = class_names[true_label]

pred_breed, confidence = predict_image(img_path, model, class_names, val_transform)

# Clean names for display
clean_true = ' '.join(true_breed.split('-')[1].split('_')) if '-' in true_breed else true_breed
clean_pred = ' '.join(pred_breed.split('-')[1].split('_')) if '-' in pred_breed else pred_breed

print(f"True breed: {clean_true}")
print(f"Predicted: {clean_pred}")
print(f"Confidence: {confidence:.4f} ({confidence*100:.2f}%)")